# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing data elements by their `@id` in a Croissant schema.

### Dataset Source
The dataset is described by its Croissant schema at the following URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as a Python dataclass (not subscripting as a dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review record sets and their fields, referencing by `@id`.

We'll print all available record sets and the fields within each, using their `@id`. Then we preview a small sample of records for each record set.

In [ ]:
# List record set @id's and their fields
if hasattr(meta, "record_sets") and meta.record_sets:
    print("Available record sets and fields:")
    record_set_ids = []
    for rs in meta.record_sets:
        print(f"- RecordSet @id: {rs.id}")
        record_set_ids.append(rs.id)
        if hasattr(rs, "fields") and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - Field @id: {field.id}  | name: {getattr(field, 'name', '--')}")
        print()
    # Preview a few records per set
    for rsid in record_set_ids:
        print(f"Records for RecordSet @id: {rsid}")
        for i, rec in enumerate(dataset.records(record_set=rsid)):
            if i >= 2:
                break
            print(rec)
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from each available record set into Pandas DataFrames for analysis.

All record set and field references use their `@id`.


In [ ]:
# Gather available record set @id's
if hasattr(meta, "record_sets") and meta.record_sets:
    record_sets_ids = [rs.id for rs in meta.record_sets]
else:
    record_sets_ids = []

dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Preview DataFrame columns for each loaded record set
for rsid, df in dataframes.items():
    print(f"RecordSet @id: {rsid}")
    print("Columns:", df.columns.tolist())
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply some common transformations, filtering or grouping by field @id. Below, select a numeric field by its `@id`, filter, normalize, and group results.

**Update the `numeric_field_id` and `group_field_id` below with actual field @id's from above to run concrete analyses.**

In [ ]:
# Choose a RecordSet and numeric field for demonstration
# (Manually set below; in real use, fill these by inspecting the previous output)
# Example:
# record_set_id = 'cr:AdoptionPredictorsResults'  # replace with your actual recordSet @id
# numeric_field_id = 'cr:LogLikelihood'           # replace with a real field @id with numeric data
# group_field_id = 'cr:County'                   # replace with a real field @id for grouping

record_set_id = None
numeric_field_id = None
group_field_id = None

# Attempt to assign a record set and numeric field by heuristics
for rs in getattr(meta, "record_sets", []):
    for field in getattr(rs, "fields", []):
        if getattr(field, "data_type", None) in ("schema:Float", "schema:Number", "schema:Integer"):
            record_set_id = rs.id
            numeric_field_id = field.id
            break
    if record_set_id:
        break
# Choose first available text/categorical field for grouping
if record_set_id:
    for field in getattr(rs, "fields", []):
        if getattr(field, "data_type", None) == "schema:Text":
            group_field_id = field.id
            break

if not record_set_id or not numeric_field_id or record_set_id not in dataframes:
    print("Could not auto-select numeric field. Set `record_set_id`, `numeric_field_id` manually above.")
else:
    df = dataframes[record_set_id]
    if numeric_field_id in df.columns:
        # Filter rows
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        col = filtered_df[numeric_field_id]
        norm_col = (col - col.mean()) / col.std() if col.std() != 0 else col
        filtered_df[f"{numeric_field_id}_normalized"] = norm_col
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} not found in data for RecordSet @id {record_set_id}.")

## 5. Visualization
Below is an example of plotting the field distribution and relationship (scatter/histogram) between two fields by their `@id`.

**Update `numeric_field_id` or choose additional fields according to the record set and field list above.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If previous EDA cell selected a field successfully
if record_set_id and numeric_field_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of field {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

        # Optional: scatter with another numeric field
        # Find another numeric field to plot
        other_numeric = None
        for col in df.select_dtypes(include=[float, int]):
            if col != numeric_field_id:
                other_numeric = col
                break
        if other_numeric:
            plt.figure(figsize=(6,4))
            sns.scatterplot(x=df[numeric_field_id], y=df[other_numeric])
            plt.xlabel(numeric_field_id)
            plt.ylabel(other_numeric)
            plt.title(f"{numeric_field_id} vs {other_numeric}")
            plt.show()
        else:
            print("No second numeric field for scatterplot.")
    else:
        print(f"Field {numeric_field_id} not found in DataFrame.")
else:
    print("Set `record_set_id` and `numeric_field_id` as above to visualize.")

## 6. Conclusion
In this notebook, we've shown how to explore a dataset defined using the Croissant schema and accessed by `mlcroissant`. Using entity `@id` references, we listed, loaded, analyzed, and visualized record set contents for the FAIR² dataset on adoption predictors in rangeland management. 

You can adapt this notebook for different record sets, fields, or further advanced analyses.